In [54]:
!pip install -qU langchain

In [55]:
!pip install -qU langchain-google-genai

In [56]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

In [57]:
google_api_key = userdata.get('GEMINI_API')
model = init_chat_model(
    "google_genai:gemini-2.5-flash",
    api_key = google_api_key
)

creating skill demand tool

In [58]:
!pip install -qU langchain-tavily

In [59]:
from langchain_tavily import TavilySearch
from pprint import pprint

In [ ]:
tavily_api_key = userdata.get('TAVILY_API_KEY')

In [61]:
skill_demand_tool = TavilySearch(
    max_result = 10,
    topic = "general",
    search_depth = "advanced",
    tavily_api_key = tavily_api_key
)

In [63]:
rapid_api_key = userdata.get('RAPID_API_KEY')

In [64]:
import requests
from langchain.tools import tool

In [91]:
import requests

@tool
def search_jobs(skill: str,location: str) -> list:
    """Search for jobs requiring a specific skill using JSearch API from RapidAPI."""
    print(f"\nCalling search_jobs tool")
    print(f"Searching jobs for: {skill} in {location}")

    url = "https://jsearch.p.rapidapi.com/search-v2"
    headers = {
        "x-rapidapi-key": rapid_api_key,
        "x-rapidapi-host": "jsearch.p.rapidapi.com",
        "Content-Type": "application/json"
    }
    querystring = {
        "query": f"{skill} in {location}",
        "page": "1",
        "country": "in",
        "employment_types": "INTERN,FULLTIME",
        "job_requirements": "no_experience,under_3_years_experience"
    }

    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()

    jobs = data.get("data", {}).get("jobs", [])
    print(f"Found {len(jobs)} jobs\n")

    result = []
    for job in jobs:
        result.append({
            "title": job.get("job_title"),
            "company": job.get("employer_name"),
            "location": job.get("job_city"),
            "apply_link": job.get("job_apply_link")
        })
    return result


In [93]:
system_prompt = """You are a Skill-to-Career Mapping assistant that helps students understand skill demand and find matching job opportunities.

You have access to these tools:
- skill_demand_tool: Search for industry demand, salary insights, and career trends
- search_jobs: Find actual job listings requiring specific skills

Help the student by researching the skill they ask about and finding relevant opportunities.

Present results in a clean, readable format with clear sections and proper spacing. Include all job details with apply links. Don't use markdown format."""

In [94]:
from langchain.agents import create_agent
agent = create_agent(
    model = model,
    tools = [skill_demand_tool,search_jobs],
    system_prompt = system_prompt,
)

Invoking the Agent

In [98]:
user_query = "What's the demand for generative ai in the industry and show me related job openings in India"

response = agent.invoke({
    "messages": [{"role": "user", "content": user_query}]
})
print(response["messages"][-1].content[0]['text'])


Calling search_jobs tool
Searching jobs for: generative ai in India
Found 4 jobs

Here's an overview of the demand for Generative AI and related job openings in India:

Industry Demand for Generative AI:

  The demand for Generative AI is experiencing significant growth.
  Market Valuation: The global generative AI market was valued at $15.4 billion in 2023.
  Projected Growth: It is projected to grow to $94.4 billion by the end of 2029.
  CAGR: This represents a compound annual growth rate (CAGR) of 35.3% from 2024 through 2029.
  Driving Factors:
    Rising demand for personalized content and experiences.
    Advances in AI and machine learning (ML).
    Growth in video and audio content creation.
    Proliferation of cloud computing and AI-as-a-Service (AIaaS), making generative AI accessible to smaller businesses.
  Impact: Generative AI is rapidly transforming various industries such as healthcare, manufacturing, financial services, telecommunications, retail, and media & enterta